# 13 可重現研究 — 最小可驗證流程

用松柏護理之家退伍軍人症資料示範「從零到摘要」的可重現工作流程。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## 匯入套件：只匯入這次真的會用到的東西

可重現的第一步是「附上清單」——這裡先把整份 notebook 需要的三個套件一次匯入：`Path`（處理檔案路徑，跨平台不出包）、`pandas`（讀資料、算摘要）、`numpy`（Step 4 會印出版本號，讓報告能對得上你電腦裝的版本）。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `from pathlib import Path` | 用物件化路徑取代字串路徑，`Path("data") / "x.csv"` 在 Windows/Mac/Linux 都會組出正確的斜線 |
> | `import pandas as pd` | 讀 CSV、做摘要統計的主力套件 |
> | `import numpy as np` | Step 4 會印出版本號並附進報告，讓別人重現環境時能核對 |

> 💡 匯入清單本身就是文件——別人打開 notebook 第一眼看到匯入了什麼，就知道這份分析依賴哪些套件，是 `pyproject.toml` 之外的第二層線索。

In [ ]:
# 這次 notebook 全程只需要這三個套件：Path 處理路徑、pandas 讀資料、numpy 供 Step 4 印版本號
from pathlib import Path
import pandas as pd
import numpy as np

## Step 1 — 讀取資料 + 產出摘要：可重現的最小單位

把整個分析壓縮成一個 `summary` dict：從固定的 CSV 讀進來，只用 `groupby`、`sum`、`mean` 這些**確定性運算**（deterministic，同樣輸入永遠得到同樣輸出，不像抽樣或亂數每次都不同）算出幾個關鍵數字。這個 dict 就是「別人在另一台機器上重跑一次，應該要跟你算出一模一樣的東西」。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `path = Path("data/synthetic/legionella_outbreak.csv")` | 用固定路徑鎖定輸入檔——路徑本身也是「可重現」的一部分 |
> | `df = pd.read_csv(path)` | 讀取資料，沒有任何隨機成分，行為完全可預期 |
> | `df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)` | 用固定規則（不是 not_ill 就算感染）衍生欄位，規則寫在程式碼裡，不是憑印象手動標記 |
> | `summary = {...}` | 把所有關鍵數字收進一個 dict，作為這次分析「唯一的標準答案」 |

> 💡 **為什麼強調「確定性」**：`groupby(...).ngroups`、`.sum()`、`.mean()` 都是純數學運算，跟亂數、多執行緒排序、時區等「隱形變因」無關——這正是可重現研究要追求的：拿掉所有會讓同一份程式碼、同一份資料，兩次執行結果卻不同的因素。

In [ ]:
# --- Step 1: 讀取資料並產出摘要 ---
path = Path("data/synthetic/legionella_outbreak.csv")
df = pd.read_csv(path)
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)  # 固定規則：非 not_ill 就算感染

summary = {  # 這個 dict 就是這次分析「唯一的標準答案」
    "n_residents": len(df),
    "n_zones": df.groupby(["floor", "wing"]).ngroups,
    "n_infected": int(df["infected"].sum()),
    "n_deaths": int((df["outcome"] == "dead").sum()),
    "attack_rate": f"{df['infected'].mean():.1%}",
    "cfr": f"{(df['outcome'] == 'dead').sum() / df['infected'].sum():.1%}",
}

print("=== 疫情摘要 ===")
for k, v in summary.items():
    print(f"  {k}: {v}")

print("\n→ 這個 dict 就是最小可驗證結果")
print("→ 任何人在任何機器上跑都應該得到一樣的數字")

## Step 2 — 可重現檢查清單：先確認「地基」在不在

在檢查資料邏輯之前，先確認幾個環境檔案是否存在——`uv.lock` 鎖定套件版本、資料檔案要在、`pyproject.toml` 定義專案結構、`tests/` 目錄能跑測試。這一步不是分析資料，而是分析「你的分析環境」。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `checks = {...}` | 把每一項檢查用「說明文字: 布林值」存成 dict，方便逐項印出 |
> | `_P("uv.lock").exists()` | 確認鎖定檔存在——沒有它，套件版本可能因人而異 |
> | `all_pass = all(checks.values())` | 只要有一項是 False，整體就不算「可重現」 |

> ⚠️ 這份清單只檢查「檔案在不在」，不檢查「版本對不對」——真正嚴謹的 CI（見 `.github/workflows/ci.yml`）還會執行 `uv sync` 把 `uv.lock` 裡鎖定的版本實際裝起來，再跑測試。

In [ ]:
# --- Step 2: 可重現檢查清單 ---
from pathlib import Path as _P  # 避免蓋掉上面已匯入的 Path

checks = {  # 只檢查「檔案在不在」，不檢查版本是否正確
    "uv.lock 存在": _P("uv.lock").exists(),
    "資料檔存在": _P("data/synthetic/legionella_outbreak.csv").exists(),
    "pyproject.toml 存在": _P("pyproject.toml").exists(),
    "tests/ 目錄存在": _P("tests").is_dir(),
}

print("=== 可重現檢查清單 ===")
for item, ok in checks.items():
    status = "✓" if ok else "✗"
    print(f"  [{status}] {item}")

all_pass = all(checks.values())
print(f"\n→ {'全部通過！環境可重現' if all_pass else '有項目未通過，需要修正'}")

## Step 3 — 把摘要寫進檔案：讓「這次跑出的數字」留下證據

只印在螢幕上的結果，下次要比對時就得重新執行一次。把 `summary` 存成 CSV（方便用 Excel/pandas 打開）和 JSON（保留原始型別，例如整數不會變成字串）兩份，日後只要 diff 這些檔案，就能知道結果有沒有跑掉。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `output_path.mkdir(parents=True, exist_ok=True)` | 確保輸出資料夾存在，`exist_ok=True` 讓重複執行不會報錯 |
> | `summary_df.to_csv(..., index=False)` | 存成 CSV，方便用試算表或別的程式讀 |
> | `json.dump(summary, f, ensure_ascii=False, indent=2)` | 存成 JSON，型別（int/str）不會像 CSV 一樣全部變成字串 |

> 🧭 CSV 好讀但「型別會模糊」（19 讀出來可能變成字串 "19"）；JSON 型別精確但要另外寫程式解析——兩份一起存，兼顧「人看」和「程式讀」兩種用途。

In [ ]:
# --- Step 3: 將摘要寫入 CSV ---
import json

# 方法 1: 存成 CSV
summary_df = pd.DataFrame([summary])
output_path = Path("data/processed")
output_path.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(output_path / "summary.csv", index=False)

# 方法 2: 存成 JSON（保留型別）
with open(output_path / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("=== 輸出檔案 ===")
print(f"  CSV: {output_path / 'summary.csv'}")
print(f"  JSON: {output_path / 'summary.json'}")
print("\n→ 下次驗證時，比對這些檔案就知道結果是否一致")

## Step 4 — 記錄環境版本：把「跑這份分析用的機器」寫下來

同樣的程式碼在不同版本的 Python / pandas 上，結果不保證一模一樣（套件行為偶爾會改版）。把版本號印出來、附進報告，別人重現失敗時，第一件事就是比對這張表。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `sys.version.split()[0]` | 取出乾淨的 Python 版本號（去掉編譯資訊等雜訊） |
> | `platform.platform()` | 作業系統與架構，例如 Linux/Windows、x86_64/arm64 |
> | `pd.__version__` / `np.__version__` | 關鍵套件版本，跟 `uv.lock` 裡鎖定的版本應該一致 |

> 💡 光有 `uv.lock` 還不夠完整——`uv.lock` 保證「重新安裝時裝到同樣版本」，這裡印出來的是「這次實際執行時裝的是哪個版本」，兩者原則上要一致；萬一不一致（例如手動 pip install 覆蓋過），這張表能幫你抓到。

In [ ]:
# --- Step 4: 版本資訊紀錄 ---
import sys
import platform

env_info = {  # 跟 uv.lock 鎖定的版本比對，確認「實際執行環境」與「鎖定版本」一致
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
}

print("=== 環境版本資訊 ===")
for k, v in env_info.items():
    print(f"  {k}: {v}")

print("\n→ 將版本資訊附在報告中，他人才能重現你的環境")
print("→ 用 uv.lock 可以自動鎖定所有套件版本")

## Step 5 — 亂數種子＝決定論：可重現最容易忽略的坑

前面四步完全沒有用到亂數，因為讀資料、算平均數都是確定性運算。但只要分析裡出現「隨機」——像 Ch10 訓練模型時的 `train_test_split`、Ch11 的 `torch.manual_seed`、或任何 `np.random` 抽樣——沒鎖種子就等於每次執行都是不同的實驗。這裡用一個最小範例證明：**同一顆種子 → 同樣的亂數；不設種子 → 每次都不同**。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `rng = np.random.default_rng(seed)` | 建立一個獨立的亂數產生器，`seed` 相同就會產生同樣的亂數序列 |
> | `rng.integers(0, 100, 5)` | 抽 5 個 0-99 的整數，模擬任何「隨機抽樣」的動作 |
> | `sample(42)` 呼叫兩次 | 用同一顆種子跑兩次，驗證輸出是否完全一致 |
> | `sample()`（不給 seed） | 讓亂數產生器用系統熵源初始化，每次執行都不同 |

> 🎲 **這就是為什麼 Ch10 的 `train_test_split(..., random_state=42)`、Ch11 的 `torch.manual_seed(42)` 都要手動鎖種子**——沒鎖種子，模型的訓練/測試切分、權重初始化都會每次不同，同一份程式碼兩次跑出不同的準確率，讓人誤以為程式碼壞了，其實只是忘記固定亂數。

In [ ]:
# --- Step 5: 亂數種子＝決定論 ---
def sample(seed=None):
    rng = np.random.default_rng(seed)
    return rng.integers(0, 100, 5)


print("seed=42 第一次:", sample(42))
print("seed=42 第二次:", sample(42), "→ 完全一樣（可重現）")
print("沒設種子   :", sample(), "→ 每次都不同（不可重現）")

## Step 6 — 資料欄位契約（schema contract）：搶在下游分析出錯前先擋下來

可重現不只是「這次能重跑」，也要確保「以後跑起來還是同一份資料結構」。如果衛生局的系統改了欄位名稱、`clinical_severity` 多了一個新分類、或 `age` 混進了負數，後面所有分析都會悄悄算錯，卻不會報錯。這裡重新讀一次原始資料，用 `assert` 明確寫下「我對這份資料的假設」——欄位要在、類別值要在已知範圍內、數值要合理——任何一項不成立就立刻中斷，而不是讓錯誤資料悄悄流進 Step 1 的摘要。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `REQUIRED_COLUMNS = {...}` | 列出這份分析依賴的必要欄位，當作契約的第一條 |
> | `missing_cols = REQUIRED_COLUMNS - set(raw.columns)` | 用集合差集找出缺少的欄位，一次列出所有缺漏，不用一個一個試 |
> | `assert not missing_cols, f"..."` | 缺欄位就馬上中斷，錯誤訊息直接告訴你缺了哪些 |
> | `set(raw["clinical_severity"].dropna().unique()) - VALID_SEVERITY` | 檢查類別值有沒有「跑出已知範圍」的新分類 |
> | `raw["age"].between(0, 120).all()` | 檢查數值欄位有沒有超出合理範圍（例如負數或多打一個 0） |

> ⚠️ 這種檢查在資料科學圈叫 **schema contract**（欄位契約）或 data validation——正式專案常用 `pandera`、`great_expectations` 等套件把它自動化並整合進 pipeline；這裡用最原始的 `assert` 示範核心概念，重點是「先假設資料可能會壞，再寫斷言去確認」，而不是等分析結果怪怪的才回頭找資料問題。

In [ ]:
# --- Step 6: 資料欄位契約（schema contract）---
raw = pd.read_csv(path)  # 重新讀一次原始資料，不依賴前面步驟做過的轉換

REQUIRED_COLUMNS = {
    "case_id", "age", "sex", "floor", "wing", "room",
    "clinical_severity", "outcome",
    "symptom_onset_date", "hospitalized", "lab_confirmed",
}
VALID_SEVERITY = {"not_ill", "asymptomatic", "mild", "moderate", "severe"}
VALID_OUTCOME = {"survived", "dead"}

missing_cols = REQUIRED_COLUMNS - set(raw.columns)
assert not missing_cols, f"缺少必要欄位：{missing_cols}"

unexpected_severity = set(raw["clinical_severity"].dropna().unique()) - VALID_SEVERITY
assert not unexpected_severity, f"clinical_severity 出現未知類別：{unexpected_severity}"

unexpected_outcome = set(raw["outcome"].dropna().unique()) - VALID_OUTCOME
assert not unexpected_outcome, f"outcome 出現未知類別：{unexpected_outcome}"

assert pd.api.types.is_numeric_dtype(raw["age"]), "age 欄位型別應為數值"
assert raw["age"].between(0, 120).all(), "age 出現不合理數值（超出 0-120 歲）"

print("✅ schema OK — 欄位、型別、值域都符合預期")
print(f"   欄位數：{len(raw.columns)}，資料列數：{len(raw)}")

## 小結

| 步驟 | 學到的技能 |
|------|------------|
| 讀取 + 摘要 | 用 dict 產出最小可驗證結果 |
| 檢查清單 | 確認環境檔案齊全 |
| 輸出存檔 | CSV / JSON 保存結果供比對 |
| 版本資訊 | 記錄 Python / 套件版本 |
| 亂數種子 | 用固定 seed 讓隨機過程也能重現 |
| 欄位契約 | 用 assert 及早擋下資料結構的變動 |

**可重現的三要素**：
1. **資料**：固定的輸入檔（`legionella_outbreak.csv`）+ schema contract 把關
2. **程式碼**：版本控制（git commit）
3. **環境**：鎖定套件（`uv.lock`）+ 固定亂數種子

下一章（Ch14），我們把所有技能整合成一個完整實戰案例。